# EXP 10 — Low-Resource Neural Machine Translation (English ↔ Assamese)

Fine-tunes `facebook/nllb-200-distilled-600M` (CC-BY-NC-4.0, academic/non-commercial use) on WMT 2026 Indic MT English–Assamese data and evaluates BLEU/chrF before vs after fine-tuning.

**Kaggle setup:** Settings → Accelerator → GPU (T4 x2 or P100); Internet → On. Add the WMT 2026 training data as a Kaggle Dataset (do not mix in test/reference sentences).

## 1. Setup and GPU check

In [ ]:
!pip -q install -U transformers datasets evaluate sentencepiece sacrebleu accelerate
!git clone https://github.com/Kartiken07/LOW-RESOURCE-NEURAL-MACHINE-TRANSLATION.git repo 2>/dev/null || true
%cd repo
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable the accelerator')

## 2. Inspect the downloaded data
Set the paths below to your Kaggle dataset files. Confirm which files/columns hold English and Assamese and that rows are aligned.

In [ ]:
import glob, os
for p in glob.glob('/kaggle/input/**/*', recursive=True)[:50]: print(p)

In [ ]:
import pandas as pd
# EDIT: path and column names of the English/Assamese data
TRAIN_FILE = '/kaggle/input/YOUR-DATASET/train.tsv'
SRC_COL, TGT_COL = 'en', 'as'
raw = pd.read_csv(TRAIN_FILE, sep='\t', dtype=str, keep_default_na=False, quoting=3)
print(raw.shape, raw.columns.tolist()); raw.head(5)

## 3. Preprocess, deduplicate and split 80:10:10 (seed 42)
If the data comes as two aligned text files use `--src-file/--tgt-file`. Add `--exclude <test/reference file>` to guarantee no test sentences enter training.

In [ ]:
!python src/preprocess.py --input {TRAIN_FILE} --src-col {SRC_COL} --tgt-col {TGT_COL} --out-dir data/processed

## 4. Tokenization check
Verifies language codes (`eng_Latn` → `asm_Beng`) and that source/target stay aligned.

In [ ]:
from transformers import AutoTokenizer
MODEL = 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(MODEL, src_lang='eng_Latn', tgt_lang='asm_Beng')
train = pd.read_csv('data/processed/train.csv')
for _, r in train.head(3).iterrows():
    enc = tok(r.src, text_target=r.tgt, max_length=128, truncation=True)
    print('SRC:', r.src); print('TGT:', r.tgt)
    print('src tokens:', tok.convert_ids_to_tokens(enc['input_ids'])[:12]); print('tgt tokens:', tok.convert_ids_to_tokens(enc['labels'])[:12]); print()

## 5. Smoke test on a small subset
Run this first to catch errors/OOM cheaply; then run the full training below.

In [ ]:
!python src/train.py --data-dir data/processed --output-dir smoke --max-train-rows 200 --max-val-rows 50 --epochs 1 --batch-size 2

## 6. Baseline: pretrained model before fine-tuning
Use the same `--max-samples` for baseline and fine-tuned model so both are scored on identical examples.

In [ ]:
N_EVAL = 1000  # same value for both evaluations
!python src/evaluate.py --model facebook/nllb-200-distilled-600M --label "Pretrained (before fine-tuning)" --max-samples {N_EVAL}

## 7. Fine-tune (record the actual settings and training time in the README)

In [ ]:
!python src/train.py --data-dir data/processed --output-dir nllb-en-as --epochs 2 --batch-size 4 --grad-accum 4 --lr 2e-5 --max-length 128 --seed 42

## 8. Evaluate the fine-tuned model

In [ ]:
!python src/evaluate.py --model nllb-en-as --label "Fine-tuned" --max-samples {N_EVAL}
print(pd.read_csv('results/metrics.csv'))
pd.read_csv('results/sample_translations.csv')

## 9. Demo: translate an English sentence into Assamese

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained('nllb-en-as', src_lang='eng_Latn', tgt_lang='asm_Beng')
model = AutoModelForSeq2SeqLM.from_pretrained('nllb-en-as').to('cuda' if torch.cuda.is_available() else 'cpu').eval()
def translate(text):
    b = tok(text, return_tensors='pt', truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        out = model.generate(**b, forced_bos_token_id=tok.convert_tokens_to_ids('asm_Beng'), max_new_tokens=128, num_beams=4)
    return tok.batch_decode(out, skip_special_tokens=True)[0]
while True:
    s = input('English sentence (blank to stop): ').strip()
    if not s: break
    print('Assamese:', translate(s))

## 10. Download results
Copy `results/metrics.csv` and `results/sample_translations.csv` back into the GitHub repo's `results/` folder, and add the actual scores and training time to the README.